# 01 — The Trident Backtester
## Hull MA + 3-Step Engulfing State Machine on BTC/USDT M1

---

### Strategy overview
The Trident identifies trend reversals using a 3-step engulfing candle pattern
anchored to the Hull Moving Average color cycle:

```
LONG:  BEC1 (bull engulf, Hull RED) → Hull turns GREEN → BEC2 (bull engulf, lower low)
       → BEC3 (bull engulf, close > BEC2.high, Hull GREEN) → ENTRY
SHORT: BEAC1 (bear engulf, Hull GREEN) → Hull turns RED → BEAC2 (bear engulf, higher high)
       → BEAC3 (bear engulf, close < BEAC2.low, Hull RED) → ENTRY
```

Reset rules: Hull changes color at any point during BEC2 or BEC3 search → full reset.

### Data
- Chart: `BTCUSDT_M1_clean.csv` — 1-minute bars Jan 2023 → May 2026
- HTF:   `BTCUSDT_M5_resampled.csv` — M5 bars (prepared for future HTF filter)

### Notebook structure
- Section 1: Imports & Parameters
- Section 2: Load Data & Indicators
- Section 3: Engulfing Detection
- Section 4: State Machine & Signal Generation
- Section 5: Backtest Engine
- Section 6: Results & Visualisation
- Section 7: Optimisation
- Section 8: Out-of-Sample Validation


---
## Section 1 — Imports & Parameters

All tunable values in one `PARAMS` dict. Change here, rerun Section 2+.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from itertools import product as iproduct
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Data paths ────────────────────────────────────────────────────────────────
M1_PATH = r"C:\Users\mad52\Documentos\Quant\The Trident\data\processed\BTCUSDT_M1_clean.csv"
M5_PATH = r"C:\Users\mad52\Documentos\Quant\The Trident\data\processed\BTCUSDT_M5_resampled.csv"

# ── Strategy parameters ───────────────────────────────────────────────────────
PARAMS = {
    # ── Hull MA ───────────────────────────────────────────────────────────────
    'hull_length'     : 55,        # ← TUNE THIS

    # ── Entry ─────────────────────────────────────────────────────────────────
    'entry_type'      : 'market',  # ← TUNE THIS: 'market' | 'bec3_mid' | 'bec1_level'
                                   #   market    = BEC3 close (immediate)
                                   #   bec3_mid  = BEC3 open (retracement into body)
                                   #   bec1_level= BEC1 high/low (structural retest)
    'entry_offset_atr': 0.1,       # ← TUNE THIS: entry offset as % of ATR

    # ── Stop Loss ─────────────────────────────────────────────────────────────
    'sl_method'       : 'bec3',    # ← TUNE THIS: 'bec3' | 'bec2'
                                   #   bec3 = signal candle low/high + ATR buffer
                                   #   bec2 = trap candle low/high + ATR buffer
    'sl_atr_mult'     : 1.5,       # ← TUNE THIS

    # ── Take Profit ───────────────────────────────────────────────────────────
    'min_rr'          : 2.0,       # ← TUNE THIS

    # ── Filters (Phase 2+) ────────────────────────────────────────────────────
    'use_ema200'      : False,     # ← TUNE THIS: 200 EMA trend filter
    'ema_length'      : 200,
    'use_m5_hull'     : False,     # ← TUNE THIS: M5 Hull context filter
    'min_body_atr'    : 0.5,       # ← TUNE THIS: min engulfing body size (% ATR), 0=off
    'session_filter'  : False,     # ← TUNE THIS: block 22:00-01:00 UTC + weekends

    # ── Risk ─────────────────────────────────────────────────────────────────
    'pct_risk'        : None,      # 1% equity per trade
    'account_size'    : 10_000,
    'max_daily_loss'  : 0.05,
    'max_total_loss'  : 0.1,
    'atr_period'      : 14,
    'fixed_risk_usd'    : 50.0,

    # ── ML filter (Phase 3) ───────────────────────────────────────────────────
    'use_ml_filter'   : True,     # ← TUNE THIS: enable CatBoost signal filter
    'ml_threshold'    : 0.55,     # ← TUNE THIS: min P(win) to take trade
    'ml_model_path'   : r"C:\Users\mad52\Documentos\Quant\The Trident\data\processed\trident_catboost.cbm",
}

print('✅ Setup complete')
for k, v in PARAMS.items():
    print(f'   {k:<22}: {v}')


---
## Section 2 — Load Data & Indicators

Load M1 and M5 data, compute Hull MA and ATR.

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df    = pd.read_csv(M1_PATH, index_col=0, parse_dates=True)
df_m5 = pd.read_csv(M5_PATH, index_col=0, parse_dates=True)

# Ensure UTC timezone
if df.index.tz is None:
    df.index    = df.index.tz_localize('UTC')
if df_m5.index.tz is None:
    df_m5.index = df_m5.index.tz_localize('UTC')

print(f'✅ M1 loaded  : {len(df):,} bars | {df.index[0].date()} → {df.index[-1].date()}')
print(f'✅ M5 loaded  : {len(df_m5):,} bars | {df_m5.index[0].date()} → {df_m5.index[-1].date()}')


In [ ]:
# ── Hull MA ───────────────────────────────────────────────────────────────────
def wma(series: pd.Series, length: int) -> pd.Series:
    weights = np.arange(1, length + 1, dtype=float)
    return series.rolling(length).apply(
        lambda x: np.dot(x, weights) / weights.sum(), raw=True)

def hull_ma(series: pd.Series, length: int) -> pd.Series:
    half  = max(1, length // 2)
    sqrtn = max(1, round(np.sqrt(length)))
    return wma(2 * wma(series, half) - wma(series, length), sqrtn)

hull_len      = PARAMS['hull_length']

# M1 Hull
df['hull']        = hull_ma(df['close'], hull_len)
df['hull_green']  = df['hull'] > df['hull'].shift(2)  # hull > hull[2] (Insilico)
df['hull_red']    = df['hull'] < df['hull'].shift(2)

# M5 Hull — forward-filled to M1 index
df_m5['hull_m5']     = hull_ma(df_m5['close'], hull_len)
df_m5['hull_m5_dir'] = df_m5['hull_m5'] > df_m5['hull_m5'].shift(2)
df['hull_m5']        = df_m5['hull_m5'].reindex(df.index, method='ffill')
df['hull_m5_green']  = df_m5['hull_m5_dir'].reindex(df.index, method='ffill')

# 200 EMA
df['ema200'] = df['close'].ewm(span=PARAMS['ema_length'], adjust=False).mean()

# ATR
high, low, prev_close = df['high'], df['low'], df['close'].shift(1)
tr = pd.concat([high-low, (high-prev_close).abs(), (low-prev_close).abs()], axis=1).max(axis=1)
df['atr'] = tr.rolling(PARAMS['atr_period']).mean()

# ── RSI(14) ──────────────────────────────────────────────────────────────────
delta = df['close'].diff()
gain  = delta.clip(lower=0).rolling(14).mean()
loss  = (-delta.clip(upper=0)).rolling(14).mean()
rs    = gain / (loss + 1e-8)
df['rsi'] = 100 - (100 / (1 + rs))

# ── VWAP (daily rolling) ──────────────────────────────────────────────────────
df['vwap'] = (df['close'] * df['volume']).groupby(df.index.date).cumsum() / \
              df['volume'].groupby(df.index.date).cumsum()

print(f'✅ Indicators computed')
print(f'   Hull (M1)  last : {df["hull"].iloc[-1]:.2f}  green={df["hull_green"].iloc[-1]}')
print(f'   Hull (M5)  last : {df["hull_m5"].iloc[-1]:.2f}  green={df["hull_m5_green"].iloc[-1]}')
print(f'   EMA200     last : {df["ema200"].iloc[-1]:.2f}')
print(f'   ATR(14)    last : {df["atr"].iloc[-1]:.2f}')
print(f'   RSI(14)    last : {df["rsi"].iloc[-1]:.2f}')
print(f'   VWAP       last : {df["vwap"].iloc[-1]:.2f}')


---
## Section 2b — Load ML Filter Model

Loads the CatBoost model trained in `02_features.ipynb`.
Only active when `use_ml_filter=True` in PARAMS.
The model predicts P(win) for each signal — trades below `ml_threshold` are skipped.


In [ ]:
# ── Load CatBoost model (if ML filter enabled) ────────────────────────────────
from catboost import CatBoostClassifier

ml_model = None

if PARAMS['use_ml_filter']:
    try:
        ml_model = CatBoostClassifier()
        ml_model.load_model(PARAMS['ml_model_path'])
        print(f'✅ ML model loaded from {PARAMS["ml_model_path"]}')
        print(f'   Threshold : {PARAMS["ml_threshold"]}')
        print(f'   Features  : {ml_model.feature_names_}')
    except Exception as e:
        print(f'❌ Could not load ML model: {e}')
        print('   Set use_ml_filter=False or train model in 02_features.ipynb first')
        ml_model = None
else:
    print('ℹ️  ML filter disabled — running baseline backtest')
    print('   Set use_ml_filter=True to enable CatBoost signal filter')


---
## Section 3 — Engulfing Detection

### HPotter body-based logic (original Engulfing Candles Detector)
```
Bullish: prev red + this green + close>=open[1] + close[1]>=open + body>prev body
Bearish: prev green + this red + open>=close[1] + open[1]>=close + body>prev body
```
No high/low range requirement — pure body engulfing.


In [ ]:
# ── Engulfing detection — HPotter logic ──────────────────────────────────────
prev_open  = df['open'].shift(1)
prev_close = df['close'].shift(1)
body_size  = (df['close'] - df['open']).abs()
prev_body  = (prev_close - prev_open).abs()

prev_red   = prev_open  > prev_close   # previous candle was red
prev_green = prev_close > prev_open    # previous candle was green
this_green = df['close'] > df['open']  # this candle is green
this_red   = df['open']  > df['close'] # this candle is red

df['bull_engulf'] = (
    prev_red                            &   # prev candle red
    this_green                          &   # this candle green
    (df['close'] >= prev_open)          &   # close >= prev open
    (prev_close  >= df['open'])         &   # prev close >= this open
    (body_size   >  prev_body)              # body > prev body
)

df['bear_engulf'] = (
    prev_green                          &   # prev candle green
    this_red                            &   # this candle red
    (df['open']  >= prev_close)         &   # open >= prev close
    (prev_open   >= df['close'])        &   # prev open >= this close
    (body_size   >  prev_body)              # body > prev body
)

# Optional: ATR-based minimum body size filter
if PARAMS['min_body_atr'] > 0:
    min_body = PARAMS['min_body_atr'] * df['atr']
    df['bull_engulf'] = df['bull_engulf'] & (body_size >= min_body)
    df['bear_engulf'] = df['bear_engulf'] & (body_size >= min_body)

months = (df.index[-1] - df.index[0]).days / 30
bull_count = df['bull_engulf'].sum()
bear_count = df['bear_engulf'].sum()
print(f'✅ Engulfing detection complete')
print(f'   Bull engulfing : {bull_count:,}  (~{bull_count/months:.0f}/month)')
print(f'   Bear engulfing : {bear_count:,}  (~{bear_count/months:.0f}/month)')
print(f'   Min body ATR   : {PARAMS["min_body_atr"]}')


---
## Section 4 — State Machine & Signal Generation

### State machine — 4 states per direction
```
State 0: idle — looking for BEC1/BEAC1
State 1: BEC1 stored — waiting for Hull to change color
State 2: Hull cycled — looking for BEC2/BEAC2
State 3: BEC2 stored — looking for BEC3/BEAC3
```

### Critical implementation note
All states in a single `if/else if` chain — prevents same-bar state bleed
where resetting to State 0 and re-entering State 1 on the same bar would
allow multiple Hull cycles to survive.

### Reset rules
- State 2: Hull changes back before BEC2 found → full reset
- State 3: Hull changes back before BEC3 found → full reset


In [ ]:
# ── State machine signal generation ──────────────────────────────────────────
def generate_signals(df: pd.DataFrame, params: dict) -> pd.DataFrame:
    """
    Run the Trident 3-step state machine bar by bar.
    Returns df with long_signal, short_signal columns and stored BEC levels.
    """
    df = df.copy()
    n  = len(df)

    # Pre-extract arrays for speed
    bull_eng   = df['bull_engulf'].values
    bear_eng   = df['bear_engulf'].values
    hull_green = df['hull_green'].values
    hull_red   = df['hull_red'].values
    closes     = df['close'].values
    opens      = df['open'].values
    highs      = df['high'].values
    lows       = df['low'].values

    # Optional filter arrays
    use_ema   = params['use_ema200']
    ema200    = df['ema200'].values
    use_m5    = params['use_m5_hull']
    m5_green  = df['hull_m5_green'].values
    use_sess  = params['session_filter']
    hour      = df.index.hour
    dow       = df.index.dayofweek   # 0=Mon … 6=Sun

    # Output arrays
    long_sig  = np.zeros(n, dtype=bool)
    short_sig = np.zeros(n, dtype=bool)

    # Store BEC levels for SL/entry calculation
    bec1_high = np.full(n, np.nan)
    bec1_low  = np.full(n, np.nan)
    bec2_high = np.full(n, np.nan)
    bec2_low  = np.full(n, np.nan)
    bec3_high = np.full(n, np.nan)
    bec3_low  = np.full(n, np.nan)
    bec3_open = np.full(n, np.nan)

    # ── Long state machine ────────────────────────────────────────────────────
    l_state    = 0
    l_bec1_cl  = l_bec1_lo = np.nan
    l_bec2_hi  = l_bec2_lo = np.nan

    # ── Short state machine ───────────────────────────────────────────────────
    s_state    = 0
    s_beac1_cl = s_beac1_hi = np.nan
    s_beac2_lo = s_beac2_hi = np.nan

    for i in range(2, n):

        # ── Session filter ────────────────────────────────────────────────────
        if use_sess:
            h, d = hour[i], dow[i]
            if (h >= 22 or h < 1) or d >= 5:
                continue

        # ── LONG state machine (if/else if — one block per bar) ───────────────
        if l_state == 0:
            if bull_eng[i] and closes[i] < df['hull'].values[i] and hull_red[i]:
                if not use_ema or closes[i] > ema200[i]:   # EMA filter
                    l_state   = 1
                    l_bec1_cl = closes[i]
                    l_bec1_lo = lows[i]

        elif l_state == 1:
            if hull_green[i]:
                # Hull turned green — check if this bar is already BEC2
                if (bull_eng[i] and lows[i] < l_bec1_lo and closes[i] < l_bec1_cl):
                    l_state   = 3
                    l_bec2_hi = highs[i]
                    l_bec2_lo = lows[i]
                else:
                    l_state = 2
            elif bull_eng[i] and closes[i] < df['hull'].values[i] and hull_red[i]:
                # Better BEC1 candidate — update
                if not use_ema or closes[i] > ema200[i]:
                    l_bec1_cl = closes[i]
                    l_bec1_lo = lows[i]

        elif l_state == 2:
            if hull_red[i]:
                # Hull turned red — one green cycle wasted → RESET
                l_state = 0; l_bec1_cl = l_bec1_lo = np.nan
            elif bull_eng[i] and lows[i] < l_bec1_lo and closes[i] < l_bec1_cl:
                l_state   = 3
                l_bec2_hi = highs[i]
                l_bec2_lo = lows[i]

        elif l_state == 3:
            if hull_red[i]:
                # Hull turned red before BEC3 → RESET
                l_state = 0
                l_bec1_cl = l_bec1_lo = l_bec2_hi = l_bec2_lo = np.nan
            elif (bull_eng[i] and closes[i] > l_bec2_hi
                  and closes[i] > df['hull'].values[i] and hull_green[i]):
                # Optional M5 Hull filter
                if not use_m5 or m5_green[i]:
                    long_sig[i]  = True
                    bec2_high[i] = l_bec2_hi
                    bec2_low[i]  = l_bec2_lo
                    bec3_high[i] = highs[i]
                    bec3_low[i]  = lows[i]
                    bec3_open[i] = opens[i]
                l_state = 0
                l_bec1_cl = l_bec1_lo = l_bec2_hi = l_bec2_lo = np.nan
            elif bull_eng[i] and closes[i] < df['hull'].values[i] and hull_red[i]:
                # Failed BEC3 — restart as BEC1
                if not use_ema or closes[i] > ema200[i]:
                    l_state = 1; l_bec1_cl = closes[i]; l_bec1_lo = lows[i]
                    l_bec2_hi = l_bec2_lo = np.nan
                else:
                    l_state = 0; l_bec1_cl = l_bec1_lo = l_bec2_hi = l_bec2_lo = np.nan

        # ── SHORT state machine (mirror logic) ────────────────────────────────
        if s_state == 0:
            if bear_eng[i] and closes[i] > df['hull'].values[i] and hull_green[i]:
                if not use_ema or closes[i] < ema200[i]:
                    s_state    = 1
                    s_beac1_cl = closes[i]
                    s_beac1_hi = highs[i]

        elif s_state == 1:
            if hull_red[i]:
                if (bear_eng[i] and highs[i] > s_beac1_hi):
                    s_state    = 3
                    s_beac2_lo = lows[i]
                    s_beac2_hi = highs[i]
                else:
                    s_state = 2
            elif bear_eng[i] and closes[i] > df['hull'].values[i] and hull_green[i]:
                if not use_ema or closes[i] < ema200[i]:
                    s_beac1_cl = closes[i]
                    s_beac1_hi = highs[i]

        elif s_state == 2:
            if hull_green[i]:
                s_state = 0; s_beac1_cl = s_beac1_hi = np.nan
            elif bear_eng[i] and highs[i] > s_beac1_hi:
                s_state    = 3
                s_beac2_lo = lows[i]
                s_beac2_hi = highs[i]

        elif s_state == 3:
            if hull_green[i]:
                s_state = 0
                s_beac1_cl = s_beac1_hi = s_beac2_lo = s_beac2_hi = np.nan
            elif (bear_eng[i] and closes[i] < s_beac2_lo
                  and closes[i] < df['hull'].values[i] and hull_red[i]):
                if not use_m5 or not m5_green[i]:
                    short_sig[i]  = True
                    bec2_high[i]  = s_beac2_hi
                    bec2_low[i]   = s_beac2_lo
                    bec3_high[i]  = highs[i]
                    bec3_low[i]   = lows[i]
                    bec3_open[i]  = opens[i]
                s_state = 0
                s_beac1_cl = s_beac1_hi = s_beac2_lo = s_beac2_hi = np.nan
            elif bear_eng[i] and closes[i] > df['hull'].values[i] and hull_green[i]:
                if not use_ema or closes[i] < ema200[i]:
                    s_state = 1; s_beac1_cl = closes[i]; s_beac1_hi = highs[i]
                    s_beac2_lo = s_beac2_hi = np.nan
                else:
                    s_state = 0; s_beac1_cl = s_beac1_hi = s_beac2_lo = s_beac2_hi = np.nan

    df['long_signal']  = long_sig
    df['short_signal'] = short_sig
    df['bec2_high']    = bec2_high
    df['bec2_low']     = bec2_low
    df['bec3_high']    = bec3_high
    df['bec3_low']     = bec3_low
    df['bec3_open']    = bec3_open
    return df


df = generate_signals(df, PARAMS)

months    = (df.index[-1] - df.index[0]).days / 30
long_cnt  = df['long_signal'].sum()
short_cnt = df['short_signal'].sum()
total     = long_cnt + short_cnt
print(f'✅ Signal generation complete')
print(f'   Long signals  : {long_cnt:,}  (~{long_cnt/months:.1f}/month)')
print(f'   Short signals : {short_cnt:,}  (~{short_cnt/months:.1f}/month)')
print(f'   Total         : {total:,}  (~{total/months:.1f}/month)')


---
## Section 5 — Backtest Engine

### Entry methods
```
market    : BEC3 close ± entry_offset × ATR  (immediate fill)
bec3_mid  : BEC3 open  (retracement into body — more conservative)
bec1_level: BEC1 high (long) / BEC1 low (short) — structural retest
            limit order, fills if price reaches level within fill_window bars
```

### SL methods
```
bec3: signal candle low/high + sl_atr_mult × ATR
bec2: trap candle low/high  + sl_atr_mult × ATR  (more structural)
```


In [ ]:
# ── Backtest engine ───────────────────────────────────────────────────────────
def run_backtest(df: pd.DataFrame, params: dict, ml_model=None) -> dict:
    n            = len(df)
    closes       = df['close'].values
    highs        = df['high'].values
    lows         = df['low'].values
    opens        = df['open'].values
    atrs         = df['atr'].values
    long_sig     = df['long_signal'].values
    short_sig    = df['short_signal'].values
    bec2_hi      = df['bec2_high'].values
    bec2_lo      = df['bec2_low'].values
    bec3_hi      = df['bec3_high'].values
    bec3_lo      = df['bec3_low'].values
    bec3_op      = df['bec3_open'].values
    index        = df.index

    # ── Feature arrays for ML logging ─────────────────────────────────────────
    hull_vals    = df['hull'].values
    hull_green_v = df['hull_green'].values
    hull_m5_gr_v = df['hull_m5_green'].values
    ema200_v     = df['ema200'].values
    rsi_v        = df['rsi'].values         if 'rsi'   in df.columns else np.zeros(n)
    vwap_v       = df['vwap'].values        if 'vwap'  in df.columns else np.zeros(n)
    volume_v     = df['volume'].values

    entry_type   = params['entry_type']
    sl_method    = params['sl_method']
    sl_mult      = params['sl_atr_mult']
    min_rr       = params['min_rr']
    pct_risk       = params.get('pct_risk', None)
    fixed_risk_usd = params.get('fixed_risk_usd', None)
    account_size   = params['account_size']
    max_daily    = params['max_daily_loss']
    max_total    = params['max_total_loss']
    entry_off    = params['entry_offset_atr']
    fill_window  = 10   # bars to wait for limit entry
    use_ml       = params.get('use_ml_filter', False) and ml_model is not None
    ml_threshold = params.get('ml_threshold', 0.525)

    # Feature column order must match 02_features.ipynb training
    ML_FEATURE_COLS = [
        'hull_green_m1', 'hull_green_m5', 'hull_dist_pct',
        'ema200_dist_pct', 'vwap_dist_pct', 'above_vwap',
        'bec3_body_atr', 'bec2_depth_atr', 'bec3_breakout_atr', 'sl_dist_atr',
        'rsi', 'rsi_oversold', 'rsi_overbought',
        'volume_ratio', 'atr_ratio', 'high_vol',
        'hour_utc', 'day_of_week', 'month', 'is_long',
    ]
    ml_filtered = 0   # count of trades skipped by ML filter

    balance            = float(account_size)
    daily_start        = float(account_size)
    current_day        = index[0].date()
    challenge_over     = False

    in_trade     = False
    direction    = None
    entry_price  = sl = tp = sl_dist = 0.0
    trade_open_i = 0

    pending      = False
    pend_dir     = None
    pend_limit   = 0.0
    pend_sl_dist = 0.0
    pend_sl_lvl  = 0.0
    pend_signal_i= 0

    equity  = []
    trades  = []

    for i in range(1, n):
        if challenge_over:
            equity.append(balance)
            continue

        # Daily reset
        bar_day = index[i].date()
        if bar_day != current_day:
            current_day  = bar_day
            daily_start  = balance

        # Daily / total drawdown checks
        if (balance - daily_start) / account_size <= -max_daily:
            equity.append(balance); continue
        if (balance - account_size) / account_size <= -max_total:
            challenge_over = True; equity.append(balance); continue

        close = closes[i]; high = highs[i]; low = lows[i]
        atr   = atrs[i]

        # ── Manage open trade ─────────────────────────────────────────────────
        if in_trade:
            if i == trade_open_i:   # same-bar guard
                equity.append(balance); continue

            sl_hit = (direction == 'long'  and low  <= sl) or                      (direction == 'short' and high >= sl)
            tp_hit = (direction == 'long'  and high >= tp) or                      (direction == 'short' and low  <= tp)

            if sl_hit or tp_hit:
                # Fixed dollar risk takes priority over % equity
                risk_amt = (fixed_risk_usd if fixed_risk_usd is not None
                            else balance * pct_risk)
                if sl_hit:
                    exit_px = sl
                    pnl     = 0.0 if abs(sl - entry_price) < 1e-8 else -risk_amt
                else:
                    exit_px     = tp
                    achieved_rr = abs(tp - entry_price) / sl_dist
                    pnl         = risk_amt * achieved_rr

                balance += pnl
                if (balance - account_size) / account_size <= -max_total:
                    challenge_over = True

                # Retrieve features stored at signal time
                sig_i = trade_open_i   # signal bar index
                trades.append({
                    # ── Trade metadata ────────────────────────────────────────
                    'open_time'       : index[sig_i],
                    'close_time'      : index[i],
                    'direction'       : direction,
                    'entry'           : round(entry_price, 4),
                    'exit'            : round(exit_px, 4),
                    'sl'              : round(sl, 4),
                    'tp'              : round(tp, 4),
                    'sl_dist'         : round(sl_dist, 4),
                    'pnl'             : round(pnl, 2),
                    'result'          : 'win' if pnl > 0 else ('be' if pnl == 0 else 'loss'),
                    'label'           : 1 if pnl > 0 else 0,   # ML target
                    'balance'         : round(balance, 2),
                    # ── Hull features ─────────────────────────────────────────
                    'hull_green_m1'   : int(hull_green_v[sig_i]),
                    'hull_green_m5'   : int(hull_m5_gr_v[sig_i]),
                    'hull_dist_pct'   : round((closes[sig_i] - hull_vals[sig_i])
                                              / hull_vals[sig_i] * 100, 4),
                    # ── EMA features ──────────────────────────────────────────
                    'ema200_dist_pct' : round((closes[sig_i] - ema200_v[sig_i])
                                              / ema200_v[sig_i] * 100, 4),
                    # ── Candle structure features ─────────────────────────────
                    'bec3_body_atr'   : round(abs(closes[sig_i] - opens[sig_i])
                                              / atrs[sig_i], 4),
                    'bec2_depth_atr'  : round(abs(closes[sig_i] - bec2_lo[sig_i])
                                              / atrs[sig_i], 4) if not np.isnan(bec2_lo[sig_i]) else 0,
                    'bec3_breakout_atr': round(abs(closes[sig_i] - bec2_hi[sig_i])
                                              / atrs[sig_i], 4) if not np.isnan(bec2_hi[sig_i]) else 0,
                    'sl_dist_atr'     : round(sl_dist / atrs[sig_i], 4),
                    # ── Volume features ───────────────────────────────────────
                    'volume_ratio'    : round(volume_v[sig_i] /
                                             (np.mean(volume_v[max(0,sig_i-20):sig_i]) + 1e-8), 4),
                    # ── RSI & VWAP ────────────────────────────────────────────
                    'rsi'             : round(float(rsi_v[sig_i]), 4),
                    'vwap_dist_pct'   : round((closes[sig_i] - vwap_v[sig_i])
                                              / vwap_v[sig_i] * 100, 4)
                                        if vwap_v[sig_i] != 0 else 0,
                    # ── Time features ─────────────────────────────────────────
                    'hour_utc'        : index[sig_i].hour,
                    'day_of_week'     : index[sig_i].dayofweek,
                    # ── ATR regime ────────────────────────────────────────────
                    'atr_ratio'       : round(atrs[sig_i] /
                                             (np.mean(atrs[max(0,sig_i-50):sig_i]) + 1e-8), 4),
                    # ── Hull slope — measures trend strength vs ranging ────────
                    # Wide band (hull far from hull[2]) = strong trend
                    # Narrow band = flat/ranging market
                    'hull_slope_atr'  : round(abs(hull_vals[sig_i] - hull_vals[max(0,sig_i-2)])
                                              / atrs[sig_i], 4),
                })
                in_trade = False; pending = False

            equity.append(balance); continue

        # ── Manage pending limit entry ─────────────────────────────────────────
        if pending:
            if i - pend_signal_i > fill_window:
                pending = False
                equity.append(balance); continue

            filled = False
            if pend_dir == 'long':
                if opens[i] <= pend_limit or low <= pend_limit:
                    fill_px = min(opens[i], pend_limit); filled = True
            else:
                if opens[i] >= pend_limit or high >= pend_limit:
                    fill_px = max(opens[i], pend_limit); filled = True

            if filled:
                sl_actual   = (fill_px - pend_sl_dist if pend_dir == 'long'
                               else fill_px + pend_sl_dist)
                tp_actual   = (fill_px + min_rr * pend_sl_dist if pend_dir == 'long'
                               else fill_px - min_rr * pend_sl_dist)
                rr_check    = abs(tp_actual - fill_px) / pend_sl_dist if pend_sl_dist > 0 else 0
                if rr_check >= min_rr:
                    in_trade     = True
                    direction    = pend_dir
                    entry_price  = fill_px
                    sl           = sl_actual
                    tp           = tp_actual
                    sl_dist      = pend_sl_dist
                    trade_open_i = i
                pending = False

            equity.append(balance); continue

        # ── New signal ────────────────────────────────────────────────────────
        if long_sig[i] or short_sig[i]:
            d        = 'long' if long_sig[i] else 'short'
            atr_val  = atr

            # ── ML filter ─────────────────────────────────────────────────────
            if use_ml:
                hv    = atrs[i] / (np.mean(atrs[max(0,i-50):i]) + 1e-8)
                feats = [
                    int(hull_green_v[i]),                                    # hull_green_m1
                    int(hull_m5_gr_v[i]),                                    # hull_green_m5
                    round((closes[i]-hull_vals[i])/hull_vals[i]*100, 4),    # hull_dist_pct
                    round((closes[i]-ema200_v[i])/ema200_v[i]*100, 4),      # ema200_dist_pct
                    round((closes[i]-vwap_v[i])/vwap_v[i]*100, 4) if vwap_v[i]!=0 else 0,  # vwap_dist_pct
                    int((closes[i]-vwap_v[i])/vwap_v[i]*100 > 0) if vwap_v[i]!=0 else 0,  # above_vwap
                    round(abs(closes[i]-opens[i])/atr_val, 4),              # bec3_body_atr
                    round(abs(closes[i]-bec2_lo[i])/atr_val, 4) if not np.isnan(bec2_lo[i]) else 0,  # bec2_depth_atr
                    round(abs(closes[i]-bec2_hi[i])/atr_val, 4) if not np.isnan(bec2_hi[i]) else 0,  # bec3_breakout_atr
                    0,                                                       # sl_dist_atr (computed below)
                    round(float(rsi_v[i]), 4),                              # rsi
                    int(rsi_v[i] < 35),                                     # rsi_oversold
                    int(rsi_v[i] > 65),                                     # rsi_overbought
                    round(volume_v[i]/(np.mean(volume_v[max(0,i-20):i])+1e-8), 4),  # volume_ratio
                    round(float(hv), 4),                                    # atr_ratio
                    int(hv > 1.2),                                          # high_vol
                    index[i].hour,                                          # hour_utc
                    index[i].dayofweek,                                     # day_of_week
                    index[i].month,                                         # month
                    int(d == 'long'),                                       # is_long
                ]
                prob = ml_model.predict_proba([feats])[0][1]
                if prob < ml_threshold:
                    ml_filtered += 1
                    equity.append(balance)
                    continue

            # ── Entry price ───────────────────────────────────────────────────
            if entry_type == 'market':
                ep = (closes[i] - entry_off * atr_val if d == 'long'
                      else closes[i] + entry_off * atr_val)
                is_limit = False
            elif entry_type == 'bec3_mid':
                ep = bec3_op[i]   # BEC3 open = retracement into body
                is_limit = True
            else:   # bec1_level — not stored yet, use bec3 as fallback
                ep = (closes[i] - entry_off * atr_val if d == 'long'
                      else closes[i] + entry_off * atr_val)
                is_limit = False

            # ── SL distance ───────────────────────────────────────────────────
            if sl_method == 'bec3':
                sl_d = ((closes[i] - bec3_lo[i]) + sl_mult * atr_val if d == 'long'
                        else (bec3_hi[i] - closes[i]) + sl_mult * atr_val)
            else:   # bec2
                sl_d = ((closes[i] - bec2_lo[i]) + sl_mult * atr_val if d == 'long'
                        else (bec2_hi[i] - closes[i]) + sl_mult * atr_val)
            sl_d = max(sl_d, 0.01)

            # ── RR check ─────────────────────────────────────────────────────
            tp_lvl = ep + min_rr * sl_d if d == 'long' else ep - min_rr * sl_d
            if not is_limit:
                sl_lvl = ep - sl_d if d == 'long' else ep + sl_d
                in_trade     = True
                direction    = d
                entry_price  = ep
                sl           = sl_lvl
                tp           = tp_lvl
                sl_dist      = sl_d
                trade_open_i = i
            else:
                pending       = True
                pend_dir      = d
                pend_limit    = ep
                pend_sl_dist  = sl_d
                pend_signal_i = i

        equity.append(balance)

    # ── Metrics ───────────────────────────────────────────────────────────────
    trades_df  = pd.DataFrame(trades)
    equity_s   = pd.Series(equity, index=df.index[:len(equity)])

    if len(trades_df) == 0:
        print('⚠️  No trades generated')
        return {'metrics': {}, 'trades': trades_df, 'equity': equity_s}

    wins    = trades_df[trades_df['result'] == 'win']
    losses  = trades_df[trades_df['result'] == 'loss']
    months  = (df.index[-1] - df.index[0]).days / 30

    win_rate = len(wins) / len(trades_df)
    avg_win  = wins['pnl'].mean()   if len(wins)   > 0 else 0
    avg_loss = losses['pnl'].abs().mean() if len(losses) > 0 else 0
    pf       = wins['pnl'].sum() / losses['pnl'].abs().sum() if len(losses) > 0 else np.inf
    peak     = equity_s.cummax()
    max_dd   = ((equity_s - peak) / account_size * 100).min()
    hard_floor = account_size * (1 - max_total)

    metrics = {
        'total_trades'    : len(trades_df),
        'wins'            : len(wins),
        'losses'          : len(losses),
        'win_rate_%'      : round(win_rate * 100, 2),
        'avg_win_$'       : round(avg_win,  2),
        'avg_loss_$'      : round(avg_loss, 2),
        'avg_rr'          : round(avg_win / avg_loss, 2) if avg_loss > 0 else 0,
        'profit_factor'   : round(pf, 2),
        'max_drawdown_%'  : round(max_dd, 2),
        'net_pnl_$'       : round(equity_s.iloc[-1] - account_size, 2),
        'final_balance_$' : round(equity_s.iloc[-1], 2),
        'trades_per_month': round(len(trades_df) / months, 1),
        'equity_min_$'    : round(float(equity_s.min()), 2),
        'hard_floor_$'    : round(hard_floor, 2),
        'challenge_over'  : challenge_over,
        'ml_filtered'     : ml_filtered,   # trades skipped by ML filter
        'signals_total'   : ml_filtered + len(trades_df),
        'fill_rate_%'     : round(len(trades_df) / (ml_filtered + len(trades_df)) * 100, 1)
                            if (ml_filtered + len(trades_df)) > 0 else 0,
    }
    return {'metrics': metrics, 'trades': trades_df, 'equity': equity_s}


# ── Run baseline backtest ─────────────────────────────────────────────────────
results = run_backtest(df, PARAMS, ml_model=ml_model)
m = results['metrics']
print('\n── The Trident — Backtest Results ─────────────────────────────────')
if m:
    for k, v in m.items():
        print(f'   {k:<22}: {v:,.2f}' if isinstance(v, float) else f'   {k:<22}: {v}')

# ── Save feature log for ML training ─────────────────────────────────────────
LOG_PATH = r"C:\Users\mad52\Documentos\Quant\The Trident\data\processed\trident_trade_log.csv"
if len(results['trades']) > 0:
    results['trades'].to_csv(LOG_PATH, index=False)
    print(f'\n✅ Trade log saved → {LOG_PATH}')
    print(f'   Rows     : {len(results["trades"]):,}')
    print(f'   Features : {len(results["trades"].columns)} columns')
    print(f'   Labels   : {results["trades"]["label"].value_counts().to_dict()}')


---
## Section 5b — Filtered vs Unfiltered Comparison

Run Section 5 twice — once with `use_ml_filter=False` and once with `True`.
Store results in `results_baseline` and `results_filtered` to compare.


In [ ]:
# ── Filtered vs Unfiltered comparison ────────────────────────────────────────
# Run this cell after running Section 5 with both use_ml_filter=False and True
# Store results manually:
#   results_baseline = run_backtest(df, {**PARAMS, 'use_ml_filter': False}, ml_model=ml_model)
#   results_filtered = run_backtest(df, {**PARAMS, 'use_ml_filter': True},  ml_model=ml_model)

results_baseline = run_backtest(df, {**PARAMS, 'use_ml_filter': False}, ml_model=ml_model)
results_filtered = run_backtest(df, {**PARAMS, 'use_ml_filter': True},  ml_model=ml_model)

mb = results_baseline['metrics']
mf = results_filtered['metrics']

print('── Filtered vs Unfiltered Comparison ───────────────────────────────────')
print(f'   {"Metric":<22} {"Baseline":>12} {"ML Filtered":>12} {"Delta"}')
print(f'   {"-"*58}')

compare_keys = ['total_trades','win_rate_%','profit_factor',
                'max_drawdown_%','net_pnl_$','trades_per_month',
                'signals_total','fill_rate_%']

for key in compare_keys:
    bv = mb.get(key, 'N/A')
    fv = mf.get(key, 'N/A')
    try:
        delta = float(fv) - float(bv)
        sign  = '+' if delta > 0 else ''
        delta_str = f'{sign}{delta:.2f}'
        # Colour verdict
        if key in ['win_rate_%','profit_factor','net_pnl_$']:
            verdict = '✅' if delta > 0 else '❌'
        elif key in ['max_drawdown_%']:
            verdict = '✅' if delta > 0 else '❌'
        elif key == 'total_trades':
            verdict = '📉 fewer trades (expected)'
        else:
            verdict = ''
        print(f'   {key:<22} {str(round(bv,2)):>12} {str(round(fv,2)):>12} '
              f'{delta_str:>10}  {verdict}')
    except:
        print(f'   {key:<22} {str(bv):>12} {str(fv):>12}')

# ── Equity curve comparison ───────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=results_baseline['equity'].index,
    y=results_baseline['equity'].values,
    name='Baseline (no filter)',
    line=dict(color='gray', width=1, dash='dot'),
))
fig.add_trace(go.Scatter(
    x=results_filtered['equity'].index,
    y=results_filtered['equity'].values,
    name=f'ML Filter (threshold={PARAMS["ml_threshold"]})',
    line=dict(color='#1D9E75', width=2),
))
fig.add_hline(y=PARAMS['account_size'],
              line_dash='dash', line_color='gray', annotation_text='Start')
fig.add_hline(y=PARAMS['account_size'] * (1 - PARAMS['max_total_loss']),
              line_dash='dot', line_color='red', annotation_text='Hard floor')

fig.update_layout(
    title='The Trident — Baseline vs ML Filtered Equity Curves',
    yaxis_title='Balance ($)',
    height=450,
    template='plotly_white',
    xaxis_rangeslider_visible=False,
)
fig.show()


---
## Section 5c — Challenge Simulation

### Purpose
Simulate the The 5ers High Stakes challenge using the ML-filtered strategy.
Shows month-by-month progression toward the +8% target with -5% daily
and -10% total limits enforced.

### Challenge rules (The 5ers High Stakes)
```
Account size : $10,000
Target       : +8%  → reach $10,800
Daily limit  : -5%  → can't lose more than $500/day
Total limit  : -10% → can't go below $9,000
```


In [ ]:
# ── Challenge simulation ──────────────────────────────────────────────────────
# Uses the ML-filtered results from Section 5b
# Simulates the challenge month by month showing progress toward target

def simulate_challenge(results, account_size=10_000, target_pct=0.08,
                       max_total_loss=0.10, max_daily_loss=0.05,
                       fixed_risk=50):
    """
    Walk through trades in order, enforcing challenge rules.
    Returns detailed monthly progress toward the target.
    """
    trades_df  = results['trades'].copy()
    if len(trades_df) == 0:
        print('No trades to simulate'); return

    trades_df['open_time'] = pd.to_datetime(trades_df['open_time'])
    trades_df = trades_df.sort_values('open_time').reset_index(drop=True)

    target      = account_size * (1 + target_pct)
    hard_floor  = account_size * (1 - max_total_loss)
    daily_limit = account_size * max_daily_loss

    balance     = float(account_size)
    peak        = float(account_size)
    daily_start = float(account_size)
    current_day = trades_df['open_time'].iloc[0].date()
    challenge_passed = False
    challenge_blown  = False
    passed_at_trade  = None
    passed_at_date   = None

    monthly_records = []
    current_month   = trades_df['open_time'].iloc[0].to_period('M')
    month_trades    = 0
    month_wins      = 0
    month_pnl       = 0.0
    month_start_bal = balance

    trade_log = []

    for _, trade in trades_df.iterrows():
        if challenge_passed or challenge_blown:
            break

        trade_day = trade['open_time'].date()
        trade_month = trade['open_time'].to_period('M')

        # Daily reset
        if trade_day != current_day:
            current_day  = trade_day
            daily_start  = balance

        # Monthly reset — save previous month record
        if trade_month != current_month:
            monthly_records.append({
                'month'       : str(current_month),
                'trades'      : month_trades,
                'wins'        : month_wins,
                'wr%'         : round(month_wins/month_trades*100, 1) if month_trades > 0 else 0,
                'pnl'         : round(month_pnl, 2),
                'end_balance' : round(balance, 2),
                'progress_%'  : round((balance - account_size) / (target - account_size) * 100, 1),
                'status'      : '✅ PASSED' if balance >= target else
                                '❌ BLOWN'  if balance <= hard_floor else
                                f'{(balance-account_size)/(target-account_size)*100:.0f}% to target',
            })
            current_month   = trade_month
            month_trades    = 0
            month_wins      = 0
            month_pnl       = 0.0
            month_start_bal = balance

        # Check daily limit
        if balance - daily_start <= -daily_limit:
            continue   # skip trades for rest of day

        # Apply trade P&L
        pnl      = trade['pnl']
        balance += pnl
        peak     = max(peak, balance)

        month_trades += 1
        month_pnl    += pnl
        if pnl > 0:
            month_wins += 1

        trade_log.append({
            'time'    : trade['open_time'],
            'balance' : round(balance, 2),
            'pnl'     : round(pnl, 2),
            'result'  : trade['result'],
        })

        # Check challenge outcome
        if balance >= target and not challenge_passed:
            challenge_passed = True
            passed_at_trade  = len(trade_log)
            passed_at_date   = trade['open_time']

        if balance <= hard_floor:
            challenge_blown = True

    # Save last month
    if month_trades > 0:
        monthly_records.append({
            'month'       : str(current_month),
            'trades'      : month_trades,
            'wins'        : month_wins,
            'wr%'         : round(month_wins/month_trades*100, 1),
            'pnl'         : round(month_pnl, 2),
            'end_balance' : round(balance, 2),
            'progress_%'  : round((balance - account_size) / (target - account_size) * 100, 1),
            'status'      : '✅ PASSED' if balance >= target else
                            '❌ BLOWN'  if balance <= hard_floor else
                            f'{(balance-account_size)/(target-account_size)*100:.0f}% to target',
        })

    monthly_df = pd.DataFrame(monthly_records)
    trade_equity = pd.DataFrame(trade_log)

    # ── Print summary ─────────────────────────────────────────────────────────
    print('── Challenge Simulation ─────────────────────────────────────────────────')
    print(f'   Account size  : ${account_size:,.0f}')
    print(f'   Target        : ${target:,.0f}  (+{target_pct*100:.0f}%)')
    print(f'   Hard floor    : ${hard_floor:,.0f}  (-{max_total_loss*100:.0f}%)')
    print(f'   Fixed risk    : ${fixed_risk}/trade')
    print()

    if challenge_passed:
        months_to_pass = monthly_df[monthly_df['status'] == '✅ PASSED'].index[0] + 1
        print(f'   ✅ CHALLENGE PASSED')
        print(f'   Passed on     : {passed_at_date.strftime("%Y-%m-%d")}')
        print(f'   Trades needed : {passed_at_trade}')
        print(f'   Months needed : {months_to_pass}')
    elif challenge_blown:
        print(f'   ❌ CHALLENGE BLOWN — balance hit ${hard_floor:,.0f}')
    else:
        print(f'   ⏳ Challenge in progress — final balance: ${balance:,.2f}')
        progress = (balance - account_size) / (target - account_size) * 100
        print(f'   Progress      : {progress:.1f}% toward target')

    print()
    print('── Monthly Progress ─────────────────────────────────────────────────────')
    print(monthly_df.to_string(index=False))

    # ── Equity curve with target/floor lines ─────────────────────────────────
    if len(trade_equity) > 0:
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=trade_equity['time'],
            y=trade_equity['balance'],
            mode='lines',
            line=dict(color='#1D9E75', width=2),
            name='Balance',
        ))
        fig.add_hline(y=target,       line_dash='dash',
                      line_color='gold',  annotation_text=f'Target ${target:,.0f}')
        fig.add_hline(y=account_size, line_dash='dash',
                      line_color='gray',  annotation_text=f'Start ${account_size:,.0f}')
        fig.add_hline(y=hard_floor,   line_dash='dot',
                      line_color='red',   annotation_text=f'Floor ${hard_floor:,.0f}')

        if challenge_passed:
            fig.add_vline(x=str(passed_at_date),
                          line_dash='dash', line_color='gold',
                          annotation_text='✅ Passed')

        fig.update_layout(
            title='The Trident — Challenge Simulation (ML Filtered)',
            yaxis_title='Balance ($)',
            height=450,
            template='plotly_white',
            xaxis_rangeslider_visible=False,
        )
        fig.show()

    return monthly_df, trade_equity


# ── Run challenge simulation on ML-filtered results ───────────────────────────
monthly_df, trade_equity = simulate_challenge(
    results_filtered,
    account_size   = PARAMS['account_size'],
    target_pct     = 0.08,
    max_total_loss = PARAMS['max_total_loss'],
    max_daily_loss = PARAMS['max_daily_loss'],
    fixed_risk     = PARAMS['fixed_risk_usd'],
)


In [ ]:
# ── Challenge simulation — TEST PERIOD ONLY (honest out-of-sample) ────────────
print('\n' + '='*70)
print('OUT-OF-SAMPLE CHALLENGE (Sep 2025 → May 2026 — model never saw this)')
print('='*70 + '\n')

# Filter to test period only
test_start = '2025-09-07'
trades_test = results_filtered['trades'].copy()
trades_test = trades_test[
    pd.to_datetime(trades_test['open_time']) >= test_start
].reset_index(drop=True)

# Build a results-like dict with test trades only
results_test_only = {
    'trades' : trades_test,
    'equity' : results_filtered['equity'],  # equity curve (display only)
    'metrics': results_filtered['metrics'],
}

print(f'Test period trades : {len(trades_test)}')
print(f'Test period WR     : {trades_test["label"].mean()*100:.1f}%')
print()

monthly_test, equity_test = simulate_challenge(
    results_test_only,
    account_size   = PARAMS['account_size'],
    target_pct     = 0.08,
    max_total_loss = PARAMS['max_total_loss'],
    max_daily_loss = PARAMS['max_daily_loss'],
    fixed_risk     = PARAMS['fixed_risk_usd'],
)

---
## Section 6 — Results & Visualisation

In [ ]:
# ── Equity curve ─────────────────────────────────────────────────────────────
def plot_results(results, params, title="The Trident"):
    if not results['metrics']:
        print('No trades to plot'); return

    equity   = results['equity']
    trades   = results['trades']
    m        = results['metrics']
    account  = params['account_size']

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        row_heights=[0.7, 0.3],
                        subplot_titles=['Equity Curve', 'Drawdown %'])

    # Equity curve
    fig.add_trace(go.Scatter(
        x=equity.index, y=equity.values,
        line=dict(color='#1D9E75', width=2), name='Equity'), row=1, col=1)
    fig.add_hline(y=account, line_dash='dash',
                  line_color='gray', row=1, col=1)
    fig.add_hline(y=account * (1 - params['max_total_loss']),
                  line_dash='dot', line_color='red',
                  annotation_text='Hard floor', row=1, col=1)

    # Drawdown
    peak  = equity.cummax()
    dd    = (equity - peak) / account * 100
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd.values,
        fill='tozeroy', line=dict(color='#E24B4A', width=1),
        name='Drawdown %'), row=2, col=1)

    fig.update_layout(
        title=(f"{title} | WR {m['win_rate_%']}% | "
               f"PF {m['profit_factor']} | MaxDD {m['max_drawdown_%']}% | "
               f"{m['total_trades']} trades"),
        height=500, template='plotly_white',
        showlegend=False, xaxis_rangeslider_visible=False,
    )
    fig.show()

    # Monthly P&L table
    if len(trades) > 0:
        trades['month'] = pd.to_datetime(trades['open_time']).dt.to_period('M')
        monthly = trades.groupby('month').agg(
            trades=('result','count'),
            wins  =('result', lambda x: (x=='win').sum()),
            pnl   =('pnl','sum')
        )
        monthly['wr%'] = (monthly['wins']/monthly['trades']*100).round(1)
        print('\n── Monthly breakdown ───────────────────────────────────────────')
        print(monthly[['trades','wins','wr%','pnl']].to_string())


plot_results(results, PARAMS)


---
## Section 6b — Loss Diagnosis

### Purpose
With only 44 trades and 25% win rate we need to know WHY we're losing.
Same diagnostic we used on the Sharp Turn M5:

```
Scenario A — Wrong direction (95.1% on Sharp Turn M5):
  Price keeps going against us after SL hit
  → Pattern not predictive on this timeframe
  → No SL adjustment will fix this

Scenario B — SL too tight:
  Price hits SL then reverses to TP
  → Widen SL or switch SL method
  → Pattern IS predictive, just needs more room
```

### Monthly win rate breakdown
Shows which months are systematically bad — useful for identifying
whether a regime filter (bull/bear market) would help.


In [ ]:
# ── Loss diagnosis ────────────────────────────────────────────────────────────
trades_df = results['trades'].copy()

if len(trades_df) == 0:
    print('No trades to diagnose')
else:
    losses = trades_df[trades_df['result'] == 'loss'].copy()
    POST_SL_WINDOW = 30   # ← TUNE THIS: bars after SL hit to check for reversal

    would_have_won  = 0
    continued_wrong = 0
    details         = []

    for _, trade in losses.iterrows():
        exit_idx = df.index.get_indexer([trade['close_time']], method='nearest')[0]
        future   = df.iloc[exit_idx + 1 : exit_idx + POST_SL_WINDOW + 1]
        if len(future) == 0:
            continue

        if trade['direction'] == 'long':
            reached_tp = (future['high'] >= trade['tp']).any()
        else:
            reached_tp = (future['low'] <= trade['tp']).any()

        if reached_tp:
            would_have_won += 1
            details.append('would_have_won')
        else:
            continued_wrong += 1
            details.append('continued_wrong')

    total_diag = would_have_won + continued_wrong
    pct_wrong  = continued_wrong / total_diag * 100 if total_diag > 0 else 0
    pct_tight  = would_have_won  / total_diag * 100 if total_diag > 0 else 0

    print('── Loss Diagnosis ───────────────────────────────────────────────────────')
    print(f'  Losses diagnosed         : {total_diag}')
    print(f'  Post-SL window           : {POST_SL_WINDOW} bars (~{POST_SL_WINDOW} min on M1)')
    print()
    print(f'  Scenario A — Wrong direction:')
    print(f'    Count : {continued_wrong} ({pct_wrong:.1f}%)')
    print(f'    Price continued against us → pattern not predictive')
    print()
    print(f'  Scenario B — SL too tight:')
    print(f'    Count : {would_have_won} ({pct_tight:.1f}%)')
    print(f'    Price hit TP after SL → widen SL or switch SL method')
    print()

    if pct_tight > 40:
        print('  ⚠️  VERDICT: SL too tight — try sl_atr_mult=2.5 or sl_method=bec2')
    elif pct_tight > 20:
        print('  ⚠️  MIXED: Some SL issues, mostly wrong direction')
    else:
        print('  ❌ VERDICT: Wrong direction — pattern needs refinement on M1')
        print('     → Try: longer hull_length, use_ema200=True, use_m5_hull=True')

    print()
    print('── Win rate by direction ────────────────────────────────────────────────')
    for d in ['long', 'short']:
        sub = trades_df[trades_df['direction'] == d]
        if len(sub) > 0:
            wr = (sub['result'] == 'win').mean() * 100
            pf_g = sub[sub['result']=='win']['pnl'].sum()
            pf_l = sub[sub['result']=='loss']['pnl'].abs().sum()
            pf   = pf_g / pf_l if pf_l > 0 else 0
            print(f'  {d:5s}: {len(sub):3d} trades | WR {wr:.1f}% | PF {pf:.2f}')

    print()
    print('── Win rate by month ────────────────────────────────────────────────────')
    trades_df['month'] = pd.to_datetime(trades_df['open_time']).dt.to_period('M')
    monthly = trades_df.groupby('month').agg(
        trades=('result','count'),
        wins  =('result', lambda x: (x=='win').sum()),
        pnl   =('pnl','sum')
    )
    monthly['wr%'] = (monthly['wins'] / monthly['trades'] * 100).round(1)
    print(monthly[['trades','wins','wr%','pnl']].to_string())


---
## Section 7 — Optimisation

Grid search on training set (first 80% of data).
Scored by `profit_factor × win_rate` — balances edge quality with consistency.


In [ ]:
# ── Optimisation grid ─────────────────────────────────────────────────────────
# ← TUNE THIS: adjust ranges based on baseline results

PARAM_GRID = {
    'hull_length'     : [45, 55, 65],
    'entry_type'      : ['market', 'bec3_mid'],
    'entry_offset_atr': [0.05, 0.1],
    'sl_method'       : ['bec3', 'bec2'],
    'sl_atr_mult'     : [1.0, 1.5, 2.0],
    'min_rr'          : [1.5, 2.0, 2.5],
    'use_ema200'      : [False],        # Phase 2 — enable after baseline
    'use_m5_hull'     : [False],        # Phase 2
    'min_body_atr'    : [0.0],          # Phase 2
    'session_filter'  : [False],        # Phase 2
    # Fixed params (not varied)
    'ema_length'      : [200],
    'pct_risk'        : [None],
    'fixed_risk_usd'  : [50],
    'account_size'    : [10_000],
    'max_daily_loss'  : [0.05],
    'max_total_loss'  : [0.10],
    'atr_period'      : [14],
}

# ── Train/test split ──────────────────────────────────────────────────────────
TRAIN_PCT = 0.80
split_idx  = int(len(df) * TRAIN_PCT)
df_train   = df.iloc[:split_idx].copy()
df_test    = df.iloc[split_idx:].copy()

print(f'Train: {df_train.index[0].date()} → {df_train.index[-1].date()} ({len(df_train):,} bars)')
print(f'Test : {df_test.index[0].date()}  → {df_test.index[-1].date()}  ({len(df_test):,} bars)')

# Total combinations
keys   = list(PARAM_GRID.keys())
values = list(PARAM_GRID.values())
total  = 1
for v in values: total *= len(v)
print(f'\nTotal combinations: {total:,}')


In [ ]:
# ── Run optimisation ──────────────────────────────────────────────────────────
from joblib import Parallel, delayed

def evaluate_combo(combo):
    params = dict(zip(keys, combo))
    try:
        # Regenerate signals with new hull_length if changed
        d = df_train.copy()
        if params['hull_length'] != PARAMS['hull_length']:
            d['hull']       = hull_ma(d['close'], params['hull_length'])
            d['hull_green'] = d['hull'] > d['hull'].shift(2)
            d['hull_red']   = d['hull'] < d['hull'].shift(2)
        d = generate_signals(d, params)
        m = run_backtest(d, params)['metrics']
        if not m: return None
        # Minimum quality filters
        if (m['total_trades'] < 20 or
            m['win_rate_%']   < 35 or
            m['profit_factor'] < 1.2 or
            m['max_drawdown_%'] < -9.0 or
            m['challenge_over']): return None
        score = m['profit_factor'] * (m['win_rate_%'] / 100)
        return {**params, **m, 'score': round(score, 4)}
    except:
        return None

print('Running optimisation...')
all_combos  = list(iproduct(*values))
raw_results = Parallel(n_jobs=-1, backend='loky', verbose=0)(
    delayed(evaluate_combo)(c) for c in tqdm(all_combos, desc='Optimising'))

opt_df = pd.DataFrame([r for r in raw_results if r is not None])
opt_df = opt_df.sort_values('score', ascending=False)

print(f'\n✅ Optimisation complete')
print(f'   Tested  : {total:,}')
print(f'   Passed  : {len(opt_df):,}')

if len(opt_df) > 0:
    display_cols = ['hull_length','entry_type','sl_method','sl_atr_mult','min_rr',
                    'total_trades','win_rate_%','profit_factor','max_drawdown_%',
                    'trades_per_month','score']
    print('\n── Top 10 combinations ─────────────────────────────────────────────')
    print(opt_df[display_cols].head(10).to_string(index=False))


---
## Section 8 — Out-of-Sample Validation

Top 3 training combinations validated on the held-out test set.

In [ ]:
# ── Out-of-sample validation ─────────────────────────────────────────────────
if len(opt_df) == 0:
    print('No valid combinations — loosen filter thresholds')
else:
    print(f'Test period: {df_test.index[0].date()} → {df_test.index[-1].date()}')
    print()

    for rank, (_, row) in enumerate(opt_df.head(3).iterrows(), 1):
        test_params = {k: row[k] for k in keys if k in row}

        # Recompute Hull if changed
        d_test = df_test.copy()
        if test_params['hull_length'] != PARAMS['hull_length']:
            d_test['hull']       = hull_ma(d_test['close'], test_params['hull_length'])
            d_test['hull_green'] = d_test['hull'] > d_test['hull'].shift(2)
            d_test['hull_red']   = d_test['hull'] < d_test['hull'].shift(2)
        d_test = generate_signals(d_test, test_params)
        m_test = run_backtest(d_test, test_params)['metrics']
        if not m_test: continue

        m_train = opt_df.iloc[rank-1]
        print(f'── Rank {rank} ─────────────────────────────────────────────────────────')
        print(f'   sl={test_params["sl_method"]} mult={test_params["sl_atr_mult"]} '
              f'rr={test_params["min_rr"]} entry={test_params["entry_type"]} '
              f'hull={test_params["hull_length"]}')
        print()
        compare = ['total_trades','win_rate_%','profit_factor','max_drawdown_%',
                   'trades_per_month','net_pnl_$']
        print(f'   {"Metric":<22} {"Train":>10} {"Test":>10} {"Verdict"}')
        print(f'   {"-"*52}')
        for key in compare:
            tv = m_train.get(key, 'N/A')
            te = m_test.get(key,  'N/A')
            try:
                diff = abs(float(te)-float(tv)) / (abs(float(tv))+1e-9)
                verdict = '✅ Robust' if diff < 0.30 else '⚠️  Overfit?'
            except: verdict = ''
            print(f'   {key:<22} {str(round(tv,2)):>10} {str(round(te,2)):>10} {verdict}')
        print()
